Immagina di avere a disposizoine un nodo con 4 GPU. Modifica il codice della lezione per configurare l'addestramento distribuito. Assicurati di impostare un batch size globale tale che ogni GPU processi esattamete 128 immagini per iterazione. Infine, aggiungi una riga di codice per stampare il numero di repliche attive per confermare che TensorFlow stia effettivamente vedendo tutti i dispositivi

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets

# 1. DEFINIZIONE DELLA STRATEGIA (Punto chiave 1 e 3)
# Rileva automaticamente tutte le GPU visibili
strategy = tf.distribute.MirroredStrategy()
print(f"Numero di dispositivi arruolati: {strategy.num_replicas_in_sync}") #rileva le GPU disponibili

print(tf.config.list_physical_devices())
print(tf.config.list_physical_devices('GPU'))

# 2. SCALARE IL BATCH SIZE
# Se vogliamo che ogni GPU lavori con 32 campioni (local batch),
# il batch globale deve essere 32 * numero_di_gpu (32*1).
BATCH_SIZE_PER_REPLICA = 128 #voglio che ogni batch sia di 64 campioni per ogni GPU
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
print("Global batch size:", GLOBAL_BATCH_SIZE)

# 3. PREPARAZIONE DEI DATI (tf.data raccomandato)
(train_images, train_labels), _ = datasets.mnist.load_data()
train_images = train_images.reshape(-1, 28, 28, 1).astype("float32") / 255

# Creiamo un dataset ottimizzato
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_dataset = train_dataset.shuffle(10000).batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 4. COSTRUZIONE DEL MODELLO DENTRO LO SCOPE (Punto chiave 3)
with strategy.scope():
    # Tutto ciò che viene creato qui sarà replicato su tutte le GPU
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# 5. ADDESTRAMENTO
# Keras gestirà automaticamente la distribuzione dei batch tra le GPU
model.fit(train_dataset, epochs=5)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
Numero di dispositivi arruolati: 1
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
[]
Global batch size: 64
Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9410 - loss: 0.2023
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9788 - loss: 0.0696
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.9854 - loss: 0.0471
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.9891 - loss: 0.0351
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.9922 - loss: 0.0260
